# Log-Loss Classification Analysis

## Assignment: Manual Implementation of Statistical Log-Loss (Cross-Entropy)

**Group:** 2

**Members:**
- Ali Cihan Ozdemir (Driver)
- Lohith (Navigator)

**Use Case:** Hours Studied vs. Exam Success (Pass/Fail)

---

## 1. Mathematical Derivation: Understanding Log-Loss

### The Feynman Technique: Explaining Log-Loss to a Peer

Imagine you're trying to predict whether a student will pass or fail an exam based on how many hours they studied. You want to know not just *what* the prediction is, but *how confident* the model is in its prediction.

**Log-Loss (Cross-Entropy)** is a metric that penalizes wrong predictions proportionally to how confident the model was. Here's the intuition:

- If a student actually **passed** (y=1) and your model predicted 90% probability of passing, that's great! The penalty should be small.
- If a student actually **failed** (y=0) but your model predicted 90% probability of passing, that's a huge mistake! The penalty should be massive.

This is exactly what Log-Loss does. It measures the surprise of the model when it makes predictions.

### The Sigmoid Function: The Bridge Between Linear Output and Probability

The **sigmoid function** transforms any real number into a value between 0 and 1, making it perfect for representing probability:

$$ \sigma(z) = \frac{1}{1 + e^{-z}} $$

where $z = w \cdot x + b$ is the linear output (weighted sum of features + bias).

**Why is this the bridge?**
- Linear regression outputs can be any number (negative, zero, or huge positive)
- Probabilities must be between 0 and 1
- Sigmoid squashes any input into the [0,1] range
- At z=0, sigmoid=0.5 (maximum uncertainty)
- As z approaches infinity, sigmoid approaches 1 (high confidence in class 1)
- As z approaches negative infinity, sigmoid approaches 0 (high confidence in class 0)

### The Log-Loss Formula

The **Log-Loss (Cross-Entropy)** cost function for binary classification is defined as:

$$ J(w,b) = -\frac{1}{m} \sum_{i=1}^{m} [y^{(i)} \cdot \log(p^{(i)}) + (1 - y^{(i)}) \cdot \log(1 - p^{(i)})] $$

Where:
- $m$ = number of training examples
- $y^{(i)}$ = true label (0 or 1)
- $p^{(i)}$ = predicted probability that y=1
- $w, b$ = model parameters (weights and bias)

**Key insight:** The negative sign ensures the loss is always positive (since log values are negative for probabilities less than 1).

---

## 2. Code Implementation

In [8]:
import numpy as np
import matplotlib.pyplot as plt

### 2.1 Create Synthetic Dataset

In [2]:
hours_studied = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])
passed = np.array([0, 0, 0, 0, 0, 1, 1, 1, 1, 1])

print("Dataset: Hours Studied vs Pass/Fail")
print("-" * 40)
for hours, result in zip(hours_studied, passed):
    outcome = "PASS" if result == 1 else "FAIL"
    print(f"Hours: {hours:2d} -> {outcome}")

Dataset: Hours Studied vs Pass/Fail
----------------------------------------
Hours:  1 -> FAIL
Hours:  2 -> FAIL
Hours:  3 -> FAIL
Hours:  4 -> FAIL
Hours:  5 -> FAIL
Hours:  6 -> PASS
Hours:  7 -> PASS
Hours:  8 -> PASS
Hours:  9 -> PASS
Hours: 10 -> PASS


### 2.2 Manual Sigmoid Function Implementation

In [3]:
def sigmoid(z):
    """
    Compute the sigmoid function.
    
    The sigmoid function maps any real number to the range (0, 1).
    It transforms the linear output into a probability.
    
    Parameters:
    z: Input value (can be scalar or array)
    
    Returns:
    Sigmoid activation (probability between 0 and 1)
    """
    return 1 / (1 + np.exp(-z))

# Test the sigmoid function
test_values = np.array([-10, -5, -1, 0, 1, 5, 10])
print("Sigmoid Function Tests:")
print("-" * 40)
for z in test_values:
    print(f"sigmoid({z:3d}) = {sigmoid(z):.6f}")

Sigmoid Function Tests:
----------------------------------------
sigmoid(-10) = 0.000045
sigmoid( -5) = 0.006693
sigmoid( -1) = 0.268941
sigmoid(  0) = 0.500000
sigmoid(  1) = 0.731059
sigmoid(  5) = 0.993307
sigmoid( 10) = 0.999955


### 2.3 Manual Log-Loss (Cross-Entropy) Implementation

In [11]:
def compute_log_loss(y_true, y_pred, epsilon=1e-15):
    """
    Compute the Log-Loss (Cross-Entropy) for binary classification.
    
    This function measures the performance of a classification model where
    the prediction is a probability value between 0 and 1.
    
    Parameters:
    y_true: Array of true binary labels (0 or 1)
    y_pred: Array of predicted probabilities (0 to 1)
    epsilon: Small value to clip predictions and avoid log(0) errors
    
    Returns:
    float: Log-Loss value (lower is better)
    """
    # Clip predictions to avoid log(0) which is undefined
    y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
    
    # Compute log-loss using the cross-entropy formula
    loss = -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))
    
    return loss

### 2.4 Simple Logistic Regression Model (Manual Implementation)

In [5]:
class SimpleLogisticRegression:
    """
    A simple implementation of Logistic Regression using gradient descent.
    """
    
    def __init__(self, learning_rate=0.1, n_iterations=1000):
        self.learning_rate = learning_rate
        self.n_iterations = n_iterations
        self.weights = None
        self.bias = None
    
    def fit(self, X, y):
        """Train the logistic regression model using gradient descent."""
        n_samples, n_features = X.shape
        
        # Initialize weights and bias
        self.weights = np.zeros(n_features)
        self.bias = 0
        
        # Gradient descent
        for _ in range(self.n_iterations):
            # Linear model
            linear_output = np.dot(X, self.weights) + self.bias
            # Apply sigmoid
            y_pred = sigmoid(linear_output)
            
            # Compute gradients
            dw = (1 / n_samples) * np.dot(X.T, (y_pred - y))
            db = (1 / n_samples) * np.sum(y_pred - y)
            
            # Update parameters
            self.weights -= self.learning_rate * dw
            self.bias -= self.learning_rate * db
    
    def predict_proba(self, X):
        """Predict probability of passing."""
        linear_output = np.dot(X, self.weights) + self.bias
        return sigmoid(linear_output)
    
    def predict(self, X, threshold=0.5):
        """Predict class labels."""
        return (self.predict_proba(X) >= threshold).astype(int)

# Train the model
X = hours_studied.reshape(-1, 1)
y = passed

model = SimpleLogisticRegression(learning_rate=0.5, n_iterations=1000)
model.fit(X, y)

# Make predictions
probabilities = model.predict_proba(X)
predictions = model.predict(X)

print("Model Training Complete!")
print(f"Learned Weight: {model.weights[0]:.4f}")
print(f"Learned Bias: {model.bias:.4f}")
print()
print("Predictions vs Actual:")
print("-" * 50)
for hours, actual, prob, pred in zip(hours_studied, y, probabilities, predictions):
    actual_str = "PASS" if actual == 1 else "FAIL"
    pred_str = "PASS" if pred == 1 else "FAIL"
    print(f"Hours: {hours:2d} | Actual: {actual_str} | Prob: {prob:.4f} | Pred: {pred_str}")

Model Training Complete!
Learned Weight: 2.0026
Learned Bias: -10.8539

Predictions vs Actual:
--------------------------------------------------
Hours:  1 | Actual: FAIL | Prob: 0.0001 | Pred: FAIL
Hours:  2 | Actual: FAIL | Prob: 0.0011 | Pred: FAIL
Hours:  3 | Actual: FAIL | Prob: 0.0078 | Pred: FAIL
Hours:  4 | Actual: FAIL | Prob: 0.0550 | Pred: FAIL
Hours:  5 | Actual: FAIL | Prob: 0.3014 | Pred: FAIL
Hours:  6 | Actual: PASS | Prob: 0.7617 | Pred: PASS
Hours:  7 | Actual: PASS | Prob: 0.9595 | Pred: PASS
Hours:  8 | Actual: PASS | Prob: 0.9943 | Pred: PASS
Hours:  9 | Actual: PASS | Prob: 0.9992 | Pred: PASS
Hours: 10 | Actual: PASS | Prob: 0.9999 | Pred: PASS


---

## 3. Visualization and Analysis

### 3.1 Plot: Predicted Probability vs. Actual Result

In [6]:
plt.figure(figsize=(12, 5))

# Plot 1: Scatter plot of actual vs predicted
plt.subplot(1, 2, 1)
plt.scatter(hours_studied, y, color='red', s=100, zorder=5, label='Actual (0=Fail, 1=Pass)', edgecolors='black')
plt.plot(hours_studied, probabilities, 'b-o', linewidth=2, markersize=6, label='Predicted Probability')
plt.axhline(y=0.5, color='green', linestyle='--', alpha=0.7, label='Decision Boundary (0.5)')
plt.xlabel('Hours Studied')
plt.ylabel('Probability of Passing')
plt.title('Predicted Probability vs Actual Result')
plt.legend()
plt.grid(True, alpha=0.3)
plt.ylim(-0.1, 1.1)

# Plot 2: Bar chart comparing actual vs predicted
plt.subplot(1, 2, 2)
x_pos = np.arange(len(hours_studied))
width = 0.35
plt.bar(x_pos - width/2, y, width, label='Actual', color='red', alpha=0.7)
plt.bar(x_pos + width/2, probabilities, width, label='Predicted Prob', color='blue', alpha=0.7)
plt.axhline(y=0.5, color='green', linestyle='--', alpha=0.7, label='Threshold')
plt.xlabel('Student Index')
plt.ylabel('Value')
plt.title('Actual vs Predicted Probability')
plt.legend()
plt.xticks(x_pos, hours_studied)
plt.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# Calculate and display Log-Loss
logloss = compute_log_loss(y, probabilities)
print(f"\nLog-Loss (Cross-Entropy): {logloss:.4f}")


Log-Loss (Cross-Entropy): 0.0744


### 3.2 Loss Surface Visualization

In [7]:
# Create a loss surface showing how penalty grows as prediction moves away from true label
fig = plt.figure(figsize=(14, 5))

# Plot 1: Loss when true label = 1
ax1 = fig.add_subplot(1, 3, 1)
p = np.linspace(0.001, 0.999, 100)
loss_y1 = -np.log(p)  # When y=1
ax1.plot(p, loss_y1, 'b-', linewidth=2)
ax1.fill_between(p, loss_y1, alpha=0.3)
ax1.axvline(x=0.9, color='red', linestyle='--', label='Correct and Confident (p=0.9)')
ax1.axvline(x=0.1, color='orange', linestyle='--', label='Wrong and Confident (p=0.1)')
ax1.set_xlabel('Predicted Probability (p)')
ax1.set_ylabel('Log-Loss Penalty')
ax1.set_title('Loss Surface: True Label = 1 (Passed)')
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3)

# Plot 2: Loss when true label = 0
ax2 = fig.add_subplot(1, 3, 2)
loss_y0 = -np.log(1 - p)  # When y=0
ax2.plot(p, loss_y0, 'r-', linewidth=2)
ax2.fill_between(p, loss_y0, alpha=0.3, color='red')
ax2.axvline(x=0.1, color='green', linestyle='--', label='Correct and Confident (p=0.1)')
ax2.axvline(x=0.9, color='orange', linestyle='--', label='Wrong and Confident (p=0.9)')
ax2.set_xlabel('Predicted Probability (p)')
ax2.set_ylabel('Log-Loss Penalty')
ax2.set_title('Loss Surface: True Label = 0 (Failed)')
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

# Plot 3: 3D Loss Surface
ax3 = fig.add_subplot(1, 3, 3, projection='3d')

p_grid = np.linspace(0.001, 0.999, 50)
y_grid = np.array([0, 1])
P, Y = np.meshgrid(p_grid, y_grid)
Loss = np.zeros_like(P)
for i in range(len(y_grid)):
    for j in range(len(p_grid)):
        if y_grid[i] == 1:
            Loss[i, j] = -np.log(P[i, j])
        else:
            Loss[i, j] = -np.log(1 - P[i, j])

ax3.plot_surface(P, Y, Loss, cmap='viridis', alpha=0.8)
ax3.set_xlabel('Predicted Probability (p)')
ax3.set_ylabel('True Label (y)')
ax3.set_zlabel('Log-Loss')
ax3.set_title('3D Loss Surface')
ax3.set_yticks([0, 1])
ax3.set_yticklabels(['0 (Fail)', '1 (Pass)'])

plt.tight_layout()
plt.show()

print("\nKey Insight: Notice how the penalty grows exponentially as the prediction moves away from the true label!")
print("- When y=1 and p=0.9: Loss is small (about 0.105)")
print("- When y=1 and p=0.1: Loss is huge (about 2.303)")


Key Insight: Notice how the penalty grows exponentially as the prediction moves away from the true label!
- When y=1 and p=0.9: Loss is small (about 0.105)
- When y=1 and p=0.1: Loss is huge (about 2.303)


---

## 4. Talking Points for Presentation

### a) Why Log-Loss Replaces MSE in Classification

**Mean Squared Error (MSE)** works well for linear regression but fails in classification for three key reasons:

1. **Non-convex optimization**: MSE with sigmoid creates a non-convex cost function with multiple local minima, making gradient descent unreliable.
2. **Gradient saturation**: When predictions are very confident (close to 0 or 1), MSE gradients become extremely small, slowing learning.
3. **Poor probability calibration**: MSE does not naturally output probabilities; it outputs values that need clipping.

**Log-Loss** solves all these problems:
- Creates a convex cost function with a single global minimum
- Provides strong gradients even for confident predictions (helps fast learning)
- Naturally produces well-calibrated probabilities

### b) The Cost of Overconfidence in Model Predictions

Log-Loss heavily penalizes confident wrong predictions. This is a feature, not a bug:

- **Example**: If a student actually failed (y=0) but model predicts 99% chance of passing:
  - Loss = -log(1-0.99) = -log(0.01) = **4.605**

- **vs** a less confident wrong prediction: predicting 60% chance of passing when they failed:
  - Loss = -log(1-0.60) = -log(0.40) = **0.916**

**Why this matters**: In production systems, overconfident wrong predictions can be dangerous (medical diagnosis, fraud detection). Log-Loss forces the model to be honest about uncertainty.

This asymmetric penalty also prevents the model from just predicting 0.5 for everything (which would be safe but useless).

### c) Convergence: How Log-Loss Creates a Convex Cost Function

**Convexity** means the cost function has exactly one minimum - the global optimum. This makes optimization reliable:

1. **Mathematical proof**: Log-Loss is convex because its second derivative (Hessian) is always positive. Since sigmoid outputs are always between 0 and 1, the term is always positive.

2. **Optimization guarantee**: With gradient descent, we can be confident we will find the global minimum, not get stuck in local minima.

3. **Smooth gradients**: Unlike other loss functions, Log-Loss provides smooth, non-zero gradients throughout the range, enabling stable and fast convergence.

This is why Log-Loss is the standard choice for training classification models - it combines mathematical elegance with practical reliability.

---

## Summary

This notebook demonstrated:
- **Manual implementation** of sigmoid and Log-Loss functions from scratch
- **Visualization** of the loss surface showing exponential penalties for wrong predictions
- **Practical understanding** of why Log-Loss is superior to MSE for classification
- **Key insights** about overconfidence, convexity, and model optimization

Log-Loss (Cross-Entropy) is the fundamental loss function that powers modern machine learning classification!